# Cell 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os

%matplotlib inline
sns.set_style("whitegrid")

# Load custom modules
import sys
sys.path.append('..')
from src.data.preprocessing import EmotionDataset, prepare_data

print("Setup complete")


# Cell 2: Load and explore English data

In [ ]:
print("=" * 60)
print("ENGLISH EMOTION DATASET EDA")
print("=" * 60)

english_data = EmotionDataset(language='english')
english_data.load_csv('../data/english/train_simplified.csv')
english_data.preprocess_texts()
english_data.convert_emotions_to_ids()

df_english = english_data.data
print(f"\nDataset shape: {df_english.shape}")
print(f"\nFirst 5 examples:")
print(df_english[['text', 'emotions']].head())


# Cell 3: English emotion distribution

In [ ]:
emotions_english = []
for emotion_str in df_english['emotions']:
    emotions_english.extend(emotion_str.split(','))

emotion_counts = Counter(emotions_english)

fig, ax = plt.subplots(figsize=(10, 6))
emotions, counts = zip(*sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True))
ax.barh(emotions, counts, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('English Emotion Distribution (GoEmotions Simplified)')
plt.tight_layout()
plt.show()

print(f"\nEmotion distribution:")
for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {emotion:12s}: {count:5d}")


# Cell 4: Text length statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All data
text_lengths = df_english['text'].str.split().str.len()
axes[0].hist(text_lengths, bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Text Length Distribution (English)')
axes[0].axvline(text_lengths.mean(), color='red', linestyle='--', label=f'Mean: {text_lengths.mean():.1f}')
axes[0].legend()

# Character length
char_lengths = df_english['text'].str.len()
axes[1].hist(char_lengths, bins=50, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Character Length Distribution (English)')
axes[1].axvline(char_lengths.mean(), color='red', linestyle='--', label=f'Mean: {char_lengths.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nText Statistics:")
print(f"  Word count - Mean: {text_lengths.mean():.2f}, Median: {text_lengths.median():.2f}")
print(f"  Word count - Min: {text_lengths.min()}, Max: {text_lengths.max()}")
print(f"  Char count - Mean: {char_lengths.mean():.2f}, Median: {char_lengths.median():.2f}")
print(f"  Char count - Min: {char_lengths.min()}, Max: {char_lengths.max()}")


# Cell 5: Multi-label analysis

In [ ]:
num_emotions_per_text = df_english['emotions'].apply(lambda x: len(x.split(',')))

fig, ax = plt.subplots(figsize=(10, 6))
num_emotions_per_text.value_counts().sort_index().plot(kind='bar', ax=ax, color='seagreen')
ax.set_xlabel('Number of Emotions per Text')
ax.set_ylabel('Count')
ax.set_title('Multi-label Distribution (English)')
plt.tight_layout()
plt.show()

print(f"\nMulti-label Statistics:")
print(f"  Single emotion: {(num_emotions_per_text == 1).sum()} ({100*(num_emotions_per_text == 1).sum()/len(df_english):.1f}%)")
print(f"  Multiple emotions: {(num_emotions_per_text > 1).sum()} ({100*(num_emotions_per_text > 1).sum()/len(df_english):.1f}%)")
print(f"  Max emotions per text: {num_emotions_per_text.max()}")


# Cell 6: Load and compare with Hindi

In [ ]:
hindi_data = EmotionDataset(language='hindi')
hindi_data.load_csv('../data/hindi/train.csv')
hindi_data.preprocess_texts()
hindi_data.convert_emotions_to_ids()

df_hindi = hindi_data.data

print("=" * 60)
print("HINDI EMOTION DATASET EDA")
print("=" * 60)
print(f"\nDataset shape: {df_hindi.shape}")
print(f"\nFirst 5 examples:")
print(df_hindi[['text', 'emotions']].head())

emotions_hindi = []
for emotion_str in df_hindi['emotions']:
    emotions_hindi.extend(emotion_str.split(','))

emotion_counts_hindi = Counter(emotions_hindi)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# English
emotions_en, counts_en = zip(*sorted(Counter(emotions_english).items(), key=lambda x: x[1], reverse=True))
axes[0].barh(emotions_en, counts_en, color='steelblue')
axes[0].set_xlabel('Count')
axes[0].set_title('English Emotion Distribution')

# Hindi
emotions_hi, counts_hi = zip(*sorted(emotion_counts_hindi.items(), key=lambda x: x[1], reverse=True))
axes[1].barh(emotions_hi, counts_hi, color='coral')
axes[1].set_xlabel('Count')
axes[1].set_title('Hindi Emotion Distribution')

plt.tight_layout()
plt.show()


# Cell 7: Load and compare with Bhojpuri

In [ ]:
bhojpuri_data = EmotionDataset(language='bhojpuri')
bhojpuri_data.load_csv('../data/bhojpuri/test.csv')
bhojpuri_data.preprocess_texts()
bhojpuri_data.convert_emotions_to_ids()

df_bhojpuri = bhojpuri_data.data

print("=" * 60)
print("BHOJPURI EMOTION DATASET EDA")
print("=" * 60)
print(f"\nDataset shape: {df_bhojpuri.shape}")
print(f"\nFirst 5 examples:")
print(df_bhojpuri[['text', 'emotions']].head())

emotions_bhojpuri = []
for emotion_str in df_bhojpuri['emotions']:
    emotions_bhojpuri.extend(emotion_str.split(','))

emotion_counts_bhojpuri = Counter(emotions_bhojpuri)

print(f"\nEmotion distribution:")
for emotion, count in sorted(emotion_counts_bhojpuri.items(), key=lambda x: x[1], reverse=True):
    print(f"  {emotion:12s}: {count:5d}")


# Cell 8: Cross-lingual comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (emotion_counts, language) in zip(axes, [
    (Counter(emotions_english), 'English'),
    (emotion_counts_hindi, 'Hindi'),
    (emotion_counts_bhojpuri, 'Bhojpuri')
]):
    emotions, counts = zip(*sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    ax.barh(emotions, counts, color=colors[:len(emotions)])
    ax.set_xlabel('Count')
    ax.set_title(f'{language} Emotion Distribution')

plt.tight_layout()
plt.show()

print("Cross-lingual comparison complete")


# Cell 9: Summary statistics

In [ ]:
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

summary_data = {
    'Language': ['English', 'Hindi', 'Bhojpuri'],
    'Examples': [len(df_english), len(df_hindi), len(df_bhojpuri)],
    'Avg Words': [
        df_english['text'].str.split().str.len().mean(),
        df_hindi['text'].str.split().str.len().mean(),
        df_bhojpuri['text'].str.split().str.len().mean()
    ]
}

summary_df = pd.DataFrame(summary_data)
print(f"\n{summary_df.to_string(index=False)}")

print("\nEDA Complete!")
